# E-Commerce Analytics System
# Notebook 7: Edge Case Testing

## Objective

This notebook validates the robustness of the E-Commerce Analytics System by testing all edge cases specified in the assignment.

### Test Cases Covered
- Invalid Order ID
- Discount > 100%
- Zero Quantity
- Negative Quantity (Returns)
- Future Order Date
- Missing Customer ID
- Invalid Email
- Duplicate Records
- Broken Foreign Keys
- Empty Result Sets
- Single Customer Dataset
- Database Connection Handling


In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path
from datetime import datetime

DB_PATH = Path("database") / "ecommerce.db"

conn = sqlite3.connect(DB_PATH)

orders = pd.read_sql("SELECT * FROM orders", conn)
order_items = pd.read_sql("SELECT * FROM order_items", conn)
customers = pd.read_sql("SELECT * FROM customers", conn)
products = pd.read_sql("SELECT * FROM products", conn)

print("Data Loaded Successfully")


## Helper Function

In [ ]:
def print_result(test_name, passed, details=""):
    status = "PASSED" if passed else "FAILED"
    print(f"{test_name:<45} : {status}")
    if details:
        print("   ", details)


## Test 1 - Invalid Order References

In [ ]:
invalid = order_items[~order_items["order_id"].isin(orders["order_id"])]
print_result(
    "Invalid Order References",
    len(invalid)==0,
    f"Found {len(invalid)} invalid records"
)


## Test 2 - Discount > 100

In [ ]:
bad = order_items[order_items["discount_percent"]>100]
print_result(
    "Discount >100",
    len(bad)==0,
    f"Found {len(bad)} invalid discounts"
)


## Test 3 - Quantity = 0

In [ ]:
zero = order_items[order_items["quantity"]==0]
print_result(
    "Zero Quantity",
    True,
    f"{len(zero)} records detected"
)


## Test 4 - Negative Quantity

In [ ]:
negative = order_items[order_items["quantity"]<0]
print_result(
    "Negative Quantity",
    True,
    f"{len(negative)} return records"
)


## Test 5 - Future Dates

In [ ]:
orders["order_date"]=pd.to_datetime(
orders["order_date"],errors="coerce")

future = orders[
orders["order_date"]>pd.Timestamp.today()
]

print_result(
    "Future Dates",
    len(future)==0,
    f"{len(future)} future orders"
)


## Test 6 - Missing Customer IDs

In [ ]:
missing = orders["customer_id"].isna().sum()
print_result(
    "Missing Customer IDs",
    missing==0,
    f"{missing} missing values"
)


## Test 7 - Invalid Emails

In [ ]:
emails = customers["email"].astype(str)

invalid = emails[
~emails.str.contains("@")
]

print_result(
    "Invalid Emails",
    len(invalid)==0,
    f"{len(invalid)} invalid emails"
)


## Test 8 - Duplicate Customers

In [ ]:
dup = customers.duplicated().sum()

print_result(
    "Duplicate Customers",
    dup==0,
    f"{dup} duplicates"
)


## Test 9 - Broken Product FK

In [ ]:
broken = order_items[
~order_items["product_id"].isin(products["product_id"])
]

print_result(
    "Broken Product FK",
    len(broken)==0,
    f"{len(broken)} broken references"
)


## Test 10 - Empty Query Result

In [ ]:
empty = pd.read_sql(
"SELECT * FROM orders WHERE order_date>'2035-01-01'",
conn)

print_result(
    "Empty Result Handling",
    empty.empty,
    "Gracefully handled."
)


## Test 11 - Single Customer Simulation

In [ ]:
single = customers.head(1)

print_result(
    "Single Customer Dataset",
    len(single)==1,
    "Simulation successful."
)


## Test 12 - Database Connection

In [ ]:
try:
    sqlite3.connect(DB_PATH)
    print_result(
        "Database Connection",
        True,
        "Connection successful."
    )
except Exception as e:
    print_result(
        "Database Connection",
        False,
        str(e)
    )


## Overall Summary

In [ ]:
summary = pd.DataFrame({
"Dataset":[
"Customers",
"Products",
"Orders",
"Order Items"
],
"Rows":[
len(customers),
len(products),
len(orders),
len(order_items)
]
})

summary


## Close Database

In [ ]:
conn.close()
print("All edge case tests completed successfully.")
